# Lab 3: Agentes no Azure AI Foundry (15 min)

## Objetivos
- Criar um agente com a **Responses API** do Azure OpenAI
- Usar function calling (tools)
- Usar o Code Interpreter
- Entender o ciclo: Request → Tool Calls → Tool Outputs → Response

## Conceitos

### O que é um Agente?
Um **Agente** no Foundry é um assistente inteligente que pode:
- Usar **ferramentas** (code interpreter, file search, functions)
- Manter **contexto** ao longo de uma conversa
- **Raciocinar** sobre que ferramenta usar e quando

### Ciclo de Vida (Responses API)
```
1. Criar cliente AzureOpenAI (com endpoint e chave)
2. Enviar pedido com instruções e tools
3. Processar resposta (pode incluir tool calls)
4. Enviar resultados das tools de volta
5. Obter resposta final
```

### SDK
```python
from openai import AzureOpenAI

client = AzureOpenAI(
    azure_endpoint=FOUNDRY_ENDPOINT,
    api_key=FOUNDRY_KEY,
    api_version="2025-03-01-preview",
)
```

In [1]:
# Setup
from dotenv import load_dotenv
import os

load_dotenv("../.env")

from openai import AzureOpenAI

endpoint = os.getenv("AZURE_AI_FOUNDRY_ENDPOINT")
key = os.getenv("AZURE_AI_FOUNDRY_KEY")
model = os.getenv("MODEL_DEPLOYMENT", "gpt-4o")

client = AzureOpenAI(
    azure_endpoint=endpoint,
    api_key=key,
    api_version="2025-03-01-preview",
)

print(f"✅ Cliente conectado: {endpoint}")

✅ Cliente conectado: https://foundry-mod2.cognitiveservices.azure.com/


## 🖥️ Exercício 3.1: Agente Simples

Vamos criar um agente básico sem ferramentas - apenas com instruções.

In [2]:
# Exercício 3.1: Conversar com um agente simples via Responses API
response = client.responses.create(
    model=model,
    instructions="És um especialista em Azure. Responde de forma concisa em português de Portugal. Usa exemplos práticos.",
    input="Explica a diferença entre Azure Functions e Container Apps em 3 linhas.",
)

print(f"✅ Resposta recebida (id: {response.id})")
print(f"   Status: {response.status}\n")

# Mostrar resposta
for item in response.output:
    if item.type == "message":
        for c in item.content:
            if hasattr(c, "text"):
                print(f"🤖 {c.text}")

✅ Resposta recebida (id: resp_0606076e7cc7004b0069dd0a83881c8194bf39fd52a6c2c96c)
   Status: completed

🤖 Azure Functions é um serviço serverless orientado a eventos que executa pedaços de código (funções) de forma altamente escalável, ideal para tarefas pequenas e reactivas, como processar mensagens de uma fila.  
Azure Container Apps, por outro lado, permite executar aplicações em contentores, com maior controlo sobre escalabilidade, recursos e integração, sendo adequado para workloads mais complexos e persistentes.  
Exemplo: use Functions para processar triggers de Blob Storage e Container Apps para alojar microserviços em contentores.


## 🖥️ Exercício 3.2: Agente com Function Calling

O verdadeiro poder dos agentes é usar **ferramentas**. Vamos criar um agente que pode consultar informações.

In [3]:
# Exercício 3.2: Agente com function tools
import json

# Definir funções que o agente pode chamar
def obter_preco_servico(servico: str) -> str:
    """Retorna o preço estimado de um serviço Azure."""
    precos = {
        "app service basic": "€50/mês",
        "container apps": "€0.000012/vCPU-s",
        "functions consumption": "€0.000016/GB-s",
        "aks": "Grátis (control plane) + custo dos nodes",
        "cosmos db": "A partir de €25/mês (serverless)",
    }
    return json.dumps({"servico": servico, "preco": precos.get(servico.lower(), "Preço não disponível. Consulta azure.com/pricing")})

def obter_regiao_recomendada(caso_uso: str) -> str:
    """Recomenda a melhor região Azure para um caso de uso."""
    regioes = {
        "europa": "West Europe (Holanda) ou North Europe (Irlanda)",
        "portugal": "West Europe (mais próximo de Portugal)",
        "global": "Usa Traffic Manager com múltiplas regiões",
        "ai": "East US ou Sweden Central (melhor disponibilidade de modelos)",
    }
    return json.dumps({"caso_uso": caso_uso, "recomendacao": regioes.get(caso_uso.lower(), regioes["europa"])})

# Mapa de funções disponíveis
available_functions = {
    "obter_preco_servico": obter_preco_servico,
    "obter_regiao_recomendada": obter_regiao_recomendada,
}

# Registar as funções como tools para a Responses API
tools = [
    {
        "type": "function",
        "name": "obter_preco_servico",
        "description": "Obtém o preço estimado de um serviço Azure",
        "parameters": {
            "type": "object",
            "properties": {
                "servico": {"type": "string", "description": "Nome do serviço Azure (ex: app service basic, container apps)"}
            },
            "required": ["servico"]
        }
    },
    {
        "type": "function",
        "name": "obter_regiao_recomendada",
        "description": "Recomenda a melhor região Azure para um caso de uso",
        "parameters": {
            "type": "object",
            "properties": {
                "caso_uso": {"type": "string", "description": "Caso de uso (europa, portugal, global, ai)"}
            },
            "required": ["caso_uso"]
        }
    },
]

print("✅ Funções definidas: obter_preco_servico, obter_regiao_recomendada")

✅ Funções definidas: obter_preco_servico, obter_regiao_recomendada


In [4]:
# Criar agente com tools e testar
instructions = "És um consultor Azure. Usa as ferramentas disponíveis para dar informações precisas sobre preços e regiões. Responde em português."

# 1. Enviar pedido inicial
response = client.responses.create(
    model=model,
    instructions=instructions,
    input="Quero criar uma app com Container Apps. Quanto custa e qual a melhor região para Portugal?",
    tools=tools,
)

print(f"📨 Resposta inicial (status: {response.status})")

# 2. Processar tool calls e enviar resultados
tool_outputs = []
for item in response.output:
    if item.type == "function_call":
        func = available_functions[item.name]
        args = json.loads(item.arguments)
        result = func(**args)
        print(f"   🔧 Tool call: {item.name}({args}) → {result}")
        tool_outputs.append({"type": "function_call_output", "call_id": item.call_id, "output": result})

# 3. Enviar resultados das tools de volta ao modelo
if tool_outputs:
    response2 = client.responses.create(
        model=model,
        instructions=instructions,
        input=tool_outputs,
        tools=tools,
        previous_response_id=response.id,
    )
    print(f"\n📨 Resposta final (status: {response2.status})\n")
    for item in response2.output:
        if item.type == "message":
            for c in item.content:
                if hasattr(c, "text"):
                    print(f"🤖 {c.text}")

📨 Resposta inicial (status: completed)
   🔧 Tool call: obter_preco_servico({'servico': 'container apps'}) → {"servico": "container apps", "preco": "\u20ac0.000012/vCPU-s"}
   🔧 Tool call: obter_regiao_recomendada({'caso_uso': 'portugal'}) → {"caso_uso": "portugal", "recomendacao": "West Europe (mais pr\u00f3ximo de Portugal)"}

📨 Resposta final (status: completed)

🤖 Criar uma aplicação com Azure Container Apps tem um custo estimado de **€0.000012 por segundo de vCPU**. Este valor pode variar dependendo do tempo de execução e das especificações do serviço.

Quanto à região ideal, a **West Europe** é a melhor escolha, pois é a mais próxima de Portugal, garantindo menores latências e melhor desempenho.


## 🖥️ Exercício 3.3: Code Interpreter

O **Code Interpreter** permite ao agente escrever e executar código Python.

In [5]:
# Exercício 3.3: Agente com Code Interpreter
response_code = client.responses.create(
    model=model,
    instructions="És um analista de dados. Usa o code interpreter para fazer cálculos e criar visualizações. Responde em português.",
    input="Calcula a sequência de Fibonacci até ao 10º número e mostra o resultado numa tabela formatada.",
    tools=[{"type": "code_interpreter", "container": {"type": "auto"}}],
)

print(f"✅ Resposta recebida (status: {response_code.status})\n")
for item in response_code.output:
    if item.type == "code_interpreter_call":
        print(f"💻 Código executado:\n{item.code}\n")
    elif item.type == "message":
        for c in item.content:
            if hasattr(c, "text"):
                print(f"🤖 {c.text}")

✅ Resposta recebida (status: completed)

💻 Código executado:
import pandas as pd

# Função para calcular a sequência de Fibonacci até ao nth número
def fibonacci_sequence(n):
    sequence = [0, 1]
    for i in range(2, n):
        sequence.append(sequence[i-1] + sequence[i-2])
    return sequence[:n]

# Calculando a sequência de Fibonacci até ao 10º número
fibonacci_numbers = fibonacci_sequence(10)

# Criar uma tabela formatada com os resultados
df_fibonacci = pd.DataFrame({
    'Ordem': range(1, 11),
    'Número de Fibonacci': fibonacci_numbers
})

df_fibonacci

🤖 Aqui está a tabela formatada com os números da sequência de Fibonacci até ao 10º número:

| Ordem | Número de Fibonacci |
|-------|----------------------|
|   1   |          0          |
|   2   |          1          |
|   3   |          1          |
|   4   |          2          |
|   5   |          3          |
|   6   |          5          |
|   7   |          8          |
|   8   |         13          |
|   9   |        

In [ ]:
# Resumo das respostas criadas
print("📋 IDs das respostas criadas nesta sessão:")
print(f"   3.1 Agente simples:     {response.id[:30]}...")
print(f"   3.3 Code interpreter:   {response_code.id[:30]}...")
print("\n✅ Sessão concluída! Não há recursos persistentes para limpar.")

## ✅ Resumo

Neste lab aprendeste:
- A usar a **Responses API** do Azure OpenAI para criar agentes
- Criar agentes simples com instruções
- Usar **function calling** para dar acesso a dados externos
- Usar o **Code Interpreter** para computação

**Próximo:** [Lab 4 - Knowledge e RAG](lab04-knowledge-rag.ipynb)